<a href="https://colab.research.google.com/github/officialselun-design/pysr/blob/master/Copy_of_Sel%C3%BBn_AI_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install praat-parselmouth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 63.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import scipy.stats as stats
import scipy.signal as signal
import librosa
import parselmouth
from parselmouth.praat import call
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
import json
import warnings
warnings.filterwarnings('ignore')
class AcademicConfiguration:
    def __init__(self):
        self.sr = 16000
        self.n_fft = 2048
        self.hop_length = 512
        self.time_step = 0.01
        self.pitch_floor = 75.0
        self.pitch_ceiling = 1000.0
        self.coupling_ratio = 2.0
class PraatNonlinearExtractor:
    def __init__(self, config: AcademicConfiguration):
        self.config = config
    def analyze_signal(self, audio_path: str):
        sound = parselmouth.Sound(audio_path)
        pitch = call(sound, "To Pitch", self.config.time_step, self.config.pitch_floor, self.config.pitch_ceiling)
        harmonicity = call(sound, "To Harmonicity (cc)", self.config.time_step, self.config.pitch_floor, 0.1, 1.0)
        point_process = call(sound, "To PointProcess (periodic, cc)", self.config.pitch_floor, self.config.pitch_ceiling)
        try:
            local_jitter = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
            local_shimmer = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
            hnr = call(harmonicity, "Get mean", 0, 0)
        except Exception:
            local_jitter, local_shimmer, hnr = 0.0, 0.0, 0.0
        pitch_values = pitch.selected_array['frequency']
        pitch_values[pitch_values == 0] = np.nan
        f0_mean = np.nanmean(pitch_values)
        y, sr = librosa.load(audio_path, sr=self.config.sr)
        D = librosa.stft(y, n_fft=self.config.n_fft, hop_length=self.config.hop_length)
        S, phase = librosa.magphase(D)
        freqs = librosa.fft_frequencies(sr=sr, n_fft=self.config.n_fft)
        f1_track, f2_track, p1_track, p2_track = [], [], [], []
        for i in range(S.shape[1]):
            mags = S[:, i]
            pks, _ = signal.find_peaks(mags, height=np.max(mags)*0.1)
            pk_f = freqs[pks]
            pk_p = phase[:, i][pks]
            if len(pk_f) >= 2:
                idx = np.argsort(pk_f)
                f2_track.append(pk_f[idx][0])
                f1_track.append(pk_f[idx][-1])
                p2_track.append(np.angle(pk_p[idx][0]))
                p1_track.append(np.angle(pk_p[idx][-1]))
            else:
                fallback_f0 = f0_mean if not np.isnan(f0_mean) else 300.0
                f2_track.append(fallback_f0 / 2.0)
                f1_track.append(fallback_f0)
                p2_track.append(0.0)
                p1_track.append(0.0)
        f1, f2 = np.array(f1_track), np.array(f2_track)
        ratio = f1 / (f2 + 1e-9)
        phase_diff = np.array(p1_track) - self.config.coupling_ratio * np.array(p2_track)
        phase_wrapped = np.mod(phase_diff + np.pi, 2*np.pi) - np.pi
        pli = np.abs(np.mean(np.exp(1j * phase_wrapped)))
        entrainment_mask = np.abs(phase_wrapped) < 0.2
        time_frames = librosa.frames_to_time(np.arange(S.shape[1]), sr=sr, hop_length=self.config.hop_length)
        return {"time_frames": time_frames, "f1_hz": f1, "f2_hz": f2, "ratio_R": ratio, "pli": float(pli), "jitter": float(local_jitter), "shimmer": float(local_shimmer), "hnr": float(hnr), "f0_mean": float(f0_mean), "entrainment_mask": entrainment_mask, "spectrogram": S, "phase_wrapped": phase_wrapped}
class ScientificVisualizer:
    def __init__(self):
        self.template = "plotly_dark"
    def generate_spectrogram(self, data):
        S_db = librosa.amplitude_to_db(data["spectrogram"], ref=np.max)
        fig = go.Figure(data=go.Heatmap(z=S_db, x=data["time_frames"], y=np.linspace(0, 8000, S_db.shape[0]), colorscale='Viridis', showscale=False))
        fig.add_trace(go.Scatter(x=data["time_frames"], y=data["f1_hz"], mode='lines', name='Primary Oscillator (f1)', line=dict(color='cyan', width=1.5)))
        fig.add_trace(go.Scatter(x=data["time_frames"], y=data["f2_hz"], mode='lines', name='Secondary Oscillator (f2)', line=dict(color='red', width=1.5)))
        fig.update_layout(title="High-Resolution Spectrogram and Oscillator Tracking", template=self.template, xaxis_title="Time (s)", yaxis_title="Frequency (Hz)", height=450, margin=dict(l=40, r=40, t=40, b=40))
        return fig
    def generate_nonlinear_dashboard(self, data):
        fig = make_subplots(rows=2, cols=2, subplot_titles=("Fractional Ratio (f1/f2)", "Phase Difference (Wrapped)", "Bifurcation Distribution", "Entrainment State"))
        fig.add_trace(go.Scatter(x=data["time_frames"], y=data["ratio_R"], mode='lines', line=dict(color='orange')), row=1, col=1)
        fig.add_trace(go.Scatter(x=data["time_frames"], y=data["phase_wrapped"], mode='markers', marker=dict(color='magenta', size=3, opacity=0.5)), row=1, col=2)
        fig.add_trace(go.Histogram(x=data["ratio_R"], nbinsx=50, marker_color='purple'), row=2, col=1)
        fig.add_trace(go.Scatter(x=data["time_frames"], y=data["entrainment_mask"].astype(int), mode='lines', line=dict(color='green', shape='hv')), row=2, col=2)
        fig.update_layout(title="Nonlinear Dynamics and Phase Space Analysis", template=self.template, height=600, showlegend=False)
        return fig
class AcademicReportGenerator:
    def generate_html_report(self, data):
        status = "Stable Entrainment" if data["pli"] >= 0.7 else "Intermittent Chaos" if data["pli"] >= 0.4 else "Chaotic Uncoupled"
        border_color = "#22c55e" if data["pli"] >= 0.7 else "#f59e0b" if data["pli"] >= 0.4 else "#ef4444"
        html = f"""
        <div style="background: #0f172a; padding: 25px; border-left: 5px solid {border_color}; border-radius: 8px; color: #e2e8f0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
            <h2 style="margin-top: 0; color: #f8fafc; font-weight: 600; border-bottom: 1px solid #334155; padding-bottom: 10px;">Scientific Executive Summary</h2>
            <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-top: 20px;">
                <div style="background: #1e293b; padding: 15px; border-radius: 6px; border: 1px solid #334155;">
                    <div style="font-size: 0.85em; color: #94a3b8; text-transform: uppercase;">System Status</div>
                    <div style="font-size: 1.2em; font-weight: bold; color: {border_color}; margin-top: 5px;">{status}</div>
                </div>
                <div style="background: #1e293b; padding: 15px; border-radius: 6px; border: 1px solid #334155;">
                    <div style="font-size: 0.85em; color: #94a3b8; text-transform: uppercase;">Phase Locking Index (PLI)</div>
                    <div style="font-size: 1.2em; font-weight: bold; margin-top: 5px;">{data['pli']:.4f}</div>
                </div>
                <div style="background: #1e293b; padding: 15px; border-radius: 6px; border: 1px solid #334155;">
                    <div style="font-size: 0.85em; color: #94a3b8; text-transform: uppercase;">Harmonics-to-Noise Ratio</div>
                    <div style="font-size: 1.2em; font-weight: bold; margin-top: 5px;">{data['hnr']:.2f} dB</div>
                </div>
            </div>
            <h3 style="color: #f8fafc; font-weight: 500; margin-top: 25px;">Acoustic Perturbation Parameters (Praat Engine)</h3>
            <ul style="line-height: 1.8; color: #cbd5e1;">
                <li><b>Fundamental Frequency (F0):</b> {data['f0_mean']:.2f} Hz</li>
                <li><b>Local Jitter:</b> {data['jitter']*100:.3f} %</li>
                <li><b>Local Shimmer:</b> {data['shimmer']*100:.3f} %</li>
                <li><b>Mean Fractional Ratio (R):</b> {np.mean(data['ratio_R']):.3f} (Std: {np.std(data['ratio_R']):.3f})</li>
            </ul>
        </div>
        """
        return html
class MasterPipeline:
    def __init__(self):
        self.config = AcademicConfiguration()
        self.extractor = PraatNonlinearExtractor(self.config)
        self.viz = ScientificVisualizer()
        self.report = AcademicReportGenerator()
    def execute(self, audio_path):
        if audio_path is None: return "<div>No audio input detected.</div>", None, None, "{}"
        data = self.extractor.analyze_signal(audio_path)
        html_out = self.report.generate_html_report(data)
        fig_spec = self.viz.generate_spectrogram(data)
        fig_dash = self.viz.generate_nonlinear_dashboard(data)
        export_data = {"Acoustic_Parameters": {"F0_Mean_Hz": data["f0_mean"], "Jitter_Percent": data["jitter"]*100, "Shimmer_Percent": data["shimmer"]*100, "HNR_dB": data["hnr"]}, "Nonlinear_Dynamics": {"Phase_Locking_Index": data["pli"], "Mean_Fractional_Ratio": float(np.mean(data["ratio_R"])), "Entrainment_Probability": float(np.mean(data["entrainment_mask"]))}}
        return html_out, fig_spec, fig_dash, json.dumps(export_data, indent=4)
css_code = """
body.dark { background-color: #0b0f19; color: #e2e8f0; font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; }
.glass-container { background: rgba(15, 23, 42, 0.6) !important; backdrop-filter: blur(20px) !important; border: 1px solid rgba(255,255,255,0.08) !important; border-radius: 12px !important; }
.sidebar-panel { background: #111827 !important; padding: 25px !important; border-radius: 10px !important; border: 1px solid #1f2937 !important; }
.gradio-container { max-width: 100% !important; }
"""
def build_scientific_dashboard():
    pipeline = MasterPipeline()
    with gr.Blocks(css=css_code, theme=gr.themes.Base()) as app:
        gr.Markdown("# Selun AI: Advanced Biomechanical Acoustics and Nonlinear Dynamics Platform")
        with gr.Row():
            with gr.Column(scale=1, elem_classes="sidebar-panel"):
                gr.Markdown("### Data Acquisition")
                audio_input = gr.Audio(type="filepath", label="Audio Signal Input")
                btn_execute = gr.Button("Execute Analysis", variant="primary")
                btn_clear = gr.Button("Clear Memory")
                gr.Markdown("---")
                gr.Markdown("### Configuration Parameters")
                gr.Slider(8000, 48000, value=16000, step=1000, label="Sampling Frequency (Hz)")
                gr.Slider(512, 4096, value=2048, step=512, label="FFT Window Size")
                gr.Slider(1.0, 5.0, value=2.0, step=0.1, label="Coupling Ratio Hypothesis (k)")
            with gr.Column(scale=4):
                with gr.Tabs():
                    with gr.Tab("Executive Report"):
                        report_html = gr.HTML()
                    with gr.Tab("Spectral Analysis"):
                        plot_spectrogram = gr.Plot()
                    with gr.Tab("Phase Space & Nonlinearity"):
                        plot_dashboard = gr.Plot()
                    with gr.Tab("Data Export"):
                        json_output = gr.Textbox(label="Structured Output (JSON format)", lines=15)
                        with gr.Row():
                            gr.Button("Export JSON")
                            gr.Button("Export CSV")
        btn_execute.click(fn=pipeline.execute, inputs=[audio_input], outputs=[report_html, plot_spectrogram, plot_dashboard, json_output])
    return app
if __name__ == "__main__":
    app = build_scientific_dashboard()
    app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0406328b24ac43cbd3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
